# Clinical Genomics Variant-Calling Pipeline — Full Colab Walkthrough (Reusable)

Ye ek complete, self-contained notebook hai jo:
1. Environment set karta hai (Java, Nextflow, samtools, minimap2, Clair3-via-bioconda)
2. Pipeline repo clone karta hai
3. Reference genome aur ek chhota HiFi test read download karta hai
4. Pipeline ko **Colab-compatible** banane ke liye zaroori patches khud-ba-khud apply karta hai
   (Singularity disable, resource limits Colab ke hisaab se, Clair3 conda activation, model path)
5. Pipeline chalata hai aur results explore karta hai

**Important:** Section 4 ke patches sirf tab lagte hain jab pipeline abhi tak patch nahi hui —
dobara chalane se koi duplicate/broken edit nahi hoga (idempotent).

Beginners ke liye: har cell ko top se bottom `Shift+Enter` se chalayein.

## 1. Environment Setup

### 1.1 Java 17 (Nextflow ko chahiye)

In [ ]:
!java -version 2>/dev/null || (apt-get update -qq && apt-get install -y -qq openjdk-17-jdk)
!java -version

### 1.2 Nextflow install

In [ ]:
import shutil

if shutil.which("nextflow") is None:
    !curl -s https://get.nextflow.io | bash
    !chmod +x nextflow
    !mv nextflow /usr/local/bin/
!nextflow -version

### 1.3 samtools + minimap2 (native install)

Colab mein Docker/Singularity daemon reliably available nahi hota, is liye tools directly
system par install kar rahe hain — pipeline ko container-free chalane ke liye (Section 4
mein Singularity ko automatically disable kar denge).

In [ ]:
!apt-get install -y -qq samtools minimap2 > /dev/null
!samtools --version | head -1
!minimap2 --version

### 1.4 Miniforge (conda/mamba) + Clair3

Clair3 (variant caller) bioconda se install hota hai aur apne pre-trained models
package ke andar hi laata hai (`{CONDA_PREFIX}/bin/models/`) — is liye model ko
alag se download karne ki zaroorat nahi. Ye cell 3-5 minute le sakta hai.

In [ ]:
import os

CONDA_PREFIX = "/usr/local/miniforge"

if not os.path.exists(CONDA_PREFIX):
    !wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O miniforge.sh
    !bash miniforge.sh -b -p {CONDA_PREFIX}
!{CONDA_PREFIX}/bin/mamba --version

In [ ]:
import subprocess

env_check = subprocess.run(
    [f"{CONDA_PREFIX}/bin/mamba", "env", "list"], capture_output=True, text=True
).stdout

if "clair3" not in env_check:
    !{CONDA_PREFIX}/bin/mamba create -n clair3 -c bioconda -c conda-forge clair3 -y

!{CONDA_PREFIX}/bin/mamba run -n clair3 which run_clair3.sh
!ls {CONDA_PREFIX}/envs/clair3/bin/models/

### 1.5 Check disk, CPU, RAM

Ye numbers Section 4 ke resource-patch cell mein automatically use honge, taake pipeline
hamesha is machine ke available CPUs/RAM ke andar hi resources maange.

In [ ]:
!nproc
!free -h
!df -h .

## 2. Get the pipeline repo

In [ ]:
REPO_URL = "https://github.com/<your-username>/clinical-genomics-pipeline.git"  # <-- apna repo URL yahan daalein
import os

repo_dir = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(repo_dir):
    !git clone $REPO_URL
%cd $repo_dir

## 3. Reference genome + small HiFi test read

**Option A (default, used below)** — tiny smoke-test reference + reads from PacBio's
official tiny test-data repo. Seconds mein ready ho jata hai.

**Option B** — real GRCh38 (commented out below) — bara (~3GB), sirf real-data run ke liye.

In [ ]:
import os
os.makedirs("data", exist_ok=True)

if not os.path.exists("PacBioTestData-master"):
    !wget -q https://github.com/PacificBiosciences/PacBioTestData/archive/refs/heads/master.tar.gz -O pacbiotestdata.tar.gz
    !tar -xzf pacbiotestdata.tar.gz

# Reference genome (tiny lambda phage genome — smoke-test only)
!cp PacBioTestData-master/data/ReferenceSet/lambdaNEB/sequence/lambdaNEB.fasta data/reference.fa

# Small HiFi (CCS) test read, converted BAM -> FASTQ
CCS_BAM = "PacBioTestData-master/data/ConsensusReadSet/m130727_114215_42211_c100569412550000001823090301191423_s1_p0.ccs.bam"
!samtools fastq $CCS_BAM > data/HUCR38.fastq

!echo "Reference:" $(wc -l < data/reference.fa) "lines"
!echo "Reads:" $(( $(wc -l < data/HUCR38.fastq) / 4 ))
!ls -lh data/

In [ ]:
# Option B — real GRCh38 primary assembly reference (large, ~3GB). Uncomment to use:
# !wget https://ftp.ensembl.org/pub/release-110/fasta/homo_sapiens/dna/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz -O data/reference_GRCh38.fa.gz
# !gunzip -f data/reference_GRCh38.fa.gz

## 4. Patch the pipeline for Colab (automatic, idempotent)

Ye cell woh saari tabdeeliyan khud karta hai jo humne manual debugging mein discover ki
thin, taake koi bhi is notebook ko fresh clone par chala sake bina errors ke:

1. **Singularity disable** — Colab mein Docker/Singularity daemon reliably nahi chalta
2. **`stageInMode = 'copy'`** — Nextflow ko symlink ki bajaye real file copies staging
   karne par majboor karta hai (warna `.fai`/`.bai` files "not found" errors aati hain
   jab tools symlinks resolve karte hain)
3. **`cpus`/`memory` adjust** — module files mein hard-coded high values ko is machine
   ke available resources (Section 1.5 se) ke andar laata hai
4. **Clair3 conda activation** — `modules/variant.nf` ke andar `run_clair3.sh` se pehle
   `conda activate clair3` add karta hai (warna Nextflow ka subprocess conda env
   inherit nahi karta)
5. **Model path fix** — `/opt/models/hifi` ko bioconda-installed model folder ka symlink
   bana deta hai (bioconda install models ke sath hi aata hai, alag download ki zaroorat nahi)

In [ ]:
import os, re, subprocess

CONDA_PREFIX = "/usr/local/miniforge"
AVAILABLE_CPUS = max(1, int(subprocess.run(["nproc"], capture_output=True, text=True).stdout.strip()) - 0)
SAFE_CPUS = min(2, AVAILABLE_CPUS)  # keep it conservative for shared Colab VMs

def patch_file(path, replacements, label):
    """Apply a list of (pattern, replacement, already_applied_marker) tuples."""
    if not os.path.exists(path):
        print(f"[skip] {path} not found")
        return
    with open(path) as f:
        content = f.read()
    original = content
    for pattern, replacement, marker in replacements:
        if marker in content:
            continue  # already patched
        content = re.sub(pattern, replacement, content)
    if content != original:
        with open(path, "w") as f:
            f.write(content)
        print(f"[patched] {label} ({path})")
    else:
        print(f"[already ok] {label} ({path})")

# 1 & 2: nextflow.config — disable Singularity, force copy staging
patch_file(
    "nextflow.config",
    [
        (r"enabled\s*=\s*true(?=\s*\n\s*autoMounts)", "enabled = false", "enabled = false"),
        (
            r"(process\s*\{\s*\n\s*executor = 'local'\s*\n\s*errorStrategy = 'retry'\s*\n\s*maxRetries = 1)",
            r"\1\n    stageInMode = 'copy'",
            "stageInMode = 'copy'",
        ),
    ],
    "nextflow.config (singularity off, copy staging)",
)

# 3: cpus/memory in each module, capped to what this machine actually has
for module in ["modules/align.nf", "modules/sort.nf", "modules/variant.nf"]:
    patch_file(
        module,
        [
            (r"cpus\s+\d+", f"cpus {SAFE_CPUS}", f"cpus {SAFE_CPUS}"),
            (r"memory\s+'16 GB'", "memory '8 GB'", "memory '8 GB'"),
        ],
        f"{module} (cpus/memory capped)",
    )

# 4: Clair3 conda activation inside modules/variant.nf
patch_file(
    "modules/variant.nf",
    [
        (
            r"run_clair3\.sh \\\\",
            f"source {CONDA_PREFIX}/etc/profile.d/conda.sh && conda activate clair3 && run_clair3.sh \\\\",
            "conda activate clair3 && run_clair3.sh",
        ),
    ],
    "modules/variant.nf (conda activation)",
)

# 5: model path — symlink /opt/models/hifi to the bioconda-installed model
os.makedirs("/opt/models", exist_ok=True)
conda_model = f"{CONDA_PREFIX}/envs/clair3/bin/models/hifi"
target = "/opt/models/hifi"
if os.path.exists(conda_model) and not os.path.exists(target):
    os.symlink(conda_model, target)
    print(f"[patched] symlinked {target} -> {conda_model}")
elif os.path.exists(target):
    print(f"[already ok] {target} exists")
else:
    print(f"[warning] {conda_model} not found — check Section 1.4 ran successfully")

print("\nDone. nextflow.config and modules/*.nf are now Colab-ready.")

## 5. Run the pipeline (built-in test profile)

In [ ]:
!rm -rf work .nextflow*
!nextflow run main.nf -profile test

## 6. Run on your own real data (optional)

In [ ]:
# !nextflow run main.nf \
#     --reads data/your_reads.fastq \
#     --reference data/your_reference.fa \
#     --sample my_sample

## 7. List what got produced

In [ ]:
!find results -type f | sort

## 8. Explore the Results

Loads the final VCF file produced by Clair3 and shows a simple, beginner-friendly summary: how many variants were found, what types they are, and a preview table.

### 8.1 Locate the output VCF

In [ ]:
import gzip, os

vcf_path = "results/variants/clair3_output/merge_output.vcf.gz"
assert os.path.exists(vcf_path), f"Can't find {vcf_path} — did the pipeline finish successfully? Check Section 5's output."
print("Found:", vcf_path)

### 8.2 Parse the VCF into a table

In [ ]:
import pandas as pd

columns = ["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "SAMPLE"]
rows = []

with gzip.open(vcf_path, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        rows.append(fields[:len(columns)])

df = pd.DataFrame(rows, columns=columns)
if len(df):
    df["POS"] = df["POS"].astype(int)
print(f"Total variants found: {len(df)}")
df.head(10)

### 8.3 Classify variants as SNP vs Indel

In [ ]:
def variant_type(row):
    ref, alt = row["REF"], row["ALT"].split(",")[0]
    if len(ref) == 1 and len(alt) == 1:
        return "SNP"
    return "Indel"

if len(df):
    df["TYPE"] = df.apply(variant_type, axis=1)
    print(df["TYPE"].value_counts())
else:
    print("No variants found — expected for a tiny smoke-test dataset with a small lambda reference.")

### 8.4 Simple bar chart of variant types

In [ ]:
import matplotlib.pyplot as plt

if len(df):
    df["TYPE"].value_counts().plot(kind="bar", title="Variant types called by Clair3")
    plt.xlabel("Type")
    plt.ylabel("Count")
    plt.show()
else:
    print("Nothing to chart — no variants in this run.")

### 8.5 Variants per chromosome/contig

In [ ]:
if len(df):
    print(df["CHROM"].value_counts())
else:
    print("No variants to summarize.")

## Next steps

- Filter by `QUAL` to keep only high-confidence variants: `df[df["QUAL"].astype(float) > 20]`
- Export a filtered table: `df.to_csv("results/variants_summary.csv", index=False)`
- Load the VCF into IGV alongside `results/alignment/*.sorted.bam` for visual inspection of individual variants.
- Once the smoke-test (Option A reference, Section 3) passes end-to-end, switch to Option B
  (real GRCh38) and your real reads (`--reads` / `--reference` flags, Section 6) for actual
  clinical analysis on your four target genes (FGFR2, ACVR1, HFE, SERPINA1).